In [ ]:
 import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_excel('/content/logs_KX7A_20260212-1051 - Copy.xlsx')


In [ ]:
# Convertir Hora a formato datetime
df['Hora'] = pd.to_datetime(df['Hora'])

# Crear columna de mes
df['Mes'] = df['Hora'].dt.to_period('M')

# Crear ID de usuario (usaremos el nombre)
df['Usuario'] = df['Nombre completo del usuario']


In [ ]:
actividad = df.groupby(['Usuario','Mes']).size().reset_index(name='Eventos')


In [ ]:
tabla = actividad.pivot(index='Usuario', columns='Mes', values='Eventos').fillna(0)
tabla = tabla.sort_index(axis=1)


In [ ]:
def indicadores_evolutivos(row):

    eventos = row.values.reshape(-1,1)

    mean = np.mean(eventos)
    std = np.std(eventos)

    # Coeficiente de variación
    cv = std / mean if mean != 0 else 0

    # Tendencia (regresión lineal)
    x = np.arange(len(eventos)).reshape(-1,1)
    model = LinearRegression().fit(x, eventos)
    trend = model.coef_[0][0]

    # Dropout (caída brusca)
    diffs = np.diff(eventos.flatten())
    dropout = np.min(diffs) if len(diffs)>0 else 0

    # Periodos de inactividad
    inactivity = np.sum(eventos.flatten()==0)

    # Picos de actividad
    peak = np.sum(eventos.flatten() > (mean + std))

    return pd.Series([mean,std,cv,trend,dropout,inactivity,peak])


In [ ]:
features = tabla.apply(indicadores_evolutivos, axis=1)

features.columns = [
    'monthly_activity_mean',
    'monthly_activity_std',
    'activity_cv',
    'usage_trend',
    'dropout_rate',
    'inactivity_periods',
    'peak_activity_count'
]


In [ ]:
def clasificar(cv):

    if cv < 0.25:
        return 0   # Bajo: estable
    elif cv < 0.5:
        return 1   # Medio: fluctuaciones leves
    else:
        return 2   # Alto: cambios drásticos


In [ ]:
features['patron_uso'] = features['activity_cv'].apply(clasificar)


In [ ]:
features.head(10)


In [ ]:
top_10_mejores = features.sort_values(by='activity_cv').head(10)

top_10_mejores


In [ ]:
top_10_peores = features.sort_values(by='activity_cv', ascending=False).head(10)

top_10_peores
